# Week 06 – Feature Engineering
## Moneyball: IPL Player Valuation | Team 05
**Course:** 24ADI204 – Data Science and Visualization

### Objective
Apply feature engineering to the Week 03 cleaned IPL dataset by analysing skewness, correcting highly skewed numerical features, scaling numerical variables, and encoding categorical variables.

### Input dataset
`IPL_Moneyball_Cleaned_Week03.csv` (Week 03 cleaned dataset).

### Workflow
1. Load and inspect the cleaned dataset.
2. Calculate descriptive statistics: mean, median, variance, standard deviation, skewness and kurtosis.
3. Identify skewed features and visualize their original distributions.
4. Apply Yeo–Johnson transformation to features with absolute skewness greater than 1.
5. Compare skewness before and after transformation.
6. Apply median imputation to missing numerical values for scaling.
7. Apply Min-Max scaling and Standard scaling.
8. Apply One-Hot Encoding and Label Encoding to selected categorical features.
9. Save the three main output datasets.

**Important:** This notebook uses the cleaned Week 03 file as input. It does not overwrite that source file. Run every cell in order in Google Colab to generate actual outputs and graphs.

## 1. Import Libraries and Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer

from google.colab import files

uploaded = files.upload()

file_name = next(iter(uploaded))

df = pd.read_csv(file_name)

print("Dataset Loaded Successfully")
print("Dataset Shape:", df.shape)

display(df.head())

## 2. Dataset Information and Quality Check

In [ ]:
print("Dataset Information")
print("-------------------")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

## 3. Separate Numerical and Categorical Features

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

categorical_cols = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical Columns:")
print(numeric_cols)

print("\nCategorical Columns:")
print(categorical_cols)

print("\nNumber of Numerical Columns:", len(numeric_cols))
print("Number of Categorical Columns:", len(categorical_cols))

## 4. Descriptive Statistical Analysis

In [ ]:
stats = df[numeric_cols].describe().T

stats["Median"] = df[numeric_cols].median()
stats["Variance"] = df[numeric_cols].var()
stats["Skewness"] = df[numeric_cols].skew()
stats["Kurtosis"] = df[numeric_cols].kurt()

print("Descriptive Statistical Analysis")

display(stats.round(3))

## 5. Skewness Identification

In [ ]:
skew_values = df[numeric_cols].skew()

skew_table = pd.DataFrame({
    "Feature": skew_values.index,
    "Skewness": skew_values.values
})

skew_table["Distribution"] = np.where(
    skew_table["Skewness"] > 1,
    "Highly Right Skewed",
    np.where(
        skew_table["Skewness"] < -1,
        "Highly Left Skewed",
        "Approximately Symmetric"
    )
)

print("Skewness Analysis")

display(skew_table.round(3))

## 6. Visualize Original Distributions

In [ ]:
selected_cols = [
    col for col in ["runs", "strike_rate", "batting_avg",
                    "wickets", "economy", "final_price_lakh"]
    if col in numeric_cols
]

for col in selected_cols:
    plt.figure(figsize=(7, 4))

    sns.histplot(
        df[col].dropna(),
        kde=True
    )

    plt.title("Original Distribution - " + col)
    plt.xlabel(col)
    plt.ylabel("Frequency")

    plt.show()

## 7. Correct Skewness with Yeo–Johnson Transformation

In [ ]:
from sklearn.preprocessing import PowerTransformer

df_skew = df.copy()

skew_before = df_skew[numeric_cols].skew()

skewed_cols = skew_before[
    skew_before.abs() > 1
].index.tolist()

print("Features Selected for Skewness Correction:")
print(skewed_cols)

for col in skewed_cols:

    valid_values = df_skew[col].dropna()

    if valid_values.nunique() > 1:

        transformer = PowerTransformer(
            method="yeo-johnson",
            standardize=False
        )

        df_skew.loc[valid_values.index, col] = (
            transformer.fit_transform(
                valid_values.to_numpy().reshape(-1, 1)
            ).ravel()
        )

print("\nSkewness Correction Completed")

## 8. Compare Skewness Before and After Transformation

In [ ]:
skew_after = df_skew[numeric_cols].skew()

skew_comparison = pd.DataFrame({
    "Before Transformation": skew_before,
    "After Transformation": skew_after
})

print("Before and After Skewness Comparison")

display(skew_comparison.round(3))

## 9. Handle Missing Numerical Values for Scaling

In [ ]:
for col in skewed_cols:

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    sns.histplot(
        df[col].dropna(),
        kde=True,
        ax=axes[0]
    )

    axes[0].set_title("Before Transformation")

    sns.histplot(
        df_skew[col].dropna(),
        kde=True,
        ax=axes[1]
    )

    axes[1].set_title("After Yeo-Johnson Transformation")

    plt.suptitle("Skewness Correction: " + col)

    plt.tight_layout()
    plt.show()

## 10. Min-Max Scaling

In [ ]:
df_numeric = df_skew[numeric_cols].copy()

imputer = SimpleImputer(strategy="median")

numeric_imputed = pd.DataFrame(
    imputer.fit_transform(df_numeric),
    columns=numeric_cols,
    index=df.index
)

print("Missing Values Before Imputation:")
print(df_numeric.isnull().sum().sum())

print("\nMissing Values After Imputation:")
print(numeric_imputed.isnull().sum().sum())

## 11. Standard Scaling

In [ ]:
minmax_scaler = MinMaxScaler()

df_minmax = pd.DataFrame(
    minmax_scaler.fit_transform(numeric_imputed),
    columns=numeric_cols,
    index=df.index
)

print("Min-Max Scaled Dataset")

display(df_minmax.head())

print("\nMinimum Values:")
display(df_minmax.min().round(3))

print("\nMaximum Values:")
display(df_minmax.max().round(3))

## 12. Compare Scaling Methods

In [ ]:
standard_scaler = StandardScaler()

df_standard = pd.DataFrame(
    standard_scaler.fit_transform(numeric_imputed),
    columns=numeric_cols,
    index=df.index
)

print("Standard Scaled Dataset")

display(df_standard.head())

print("\nMean After Standard Scaling:")
display(df_standard.mean().round(3))

print("\nStandard Deviation After Scaling:")
display(df_standard.std(ddof=0).round(3))

## 13. Visual Comparison of Scaling Methods

In [ ]:
comparison = pd.DataFrame({
    "Original Mean": numeric_imputed.mean(),
    "MinMax Mean": df_minmax.mean(),
    "Standard Mean": df_standard.mean(),
    "Original Std": numeric_imputed.std(),
    "MinMax Std": df_minmax.std(),
    "Standard Std": df_standard.std(ddof=0)
})

print("Scaling Comparison")

display(comparison.round(3))

## 14. One-Hot Encoding

In [ ]:
feature = "runs" if "runs" in numeric_cols else numeric_cols[0]

plt.figure(figsize=(9, 5))

sns.kdeplot(
    numeric_imputed[feature],
    label="Original"
)

sns.kdeplot(
    df_minmax[feature],
    label="Min-Max Scaled"
)

sns.kdeplot(
    df_standard[feature],
    label="Standard Scaled"
)

plt.title("Comparison of Scaling Methods - " + feature)
plt.legend()
plt.show()

## 15. Label Encoding

In [ ]:
onehot_cols = [
    col for col in [
        "role",
        "nationality",
        "bowl_style",
        "bat_style"
    ]
    if col in categorical_cols
]

df_onehot_input = df[onehot_cols].fillna("Unknown")

df_onehot = pd.get_dummies(
    df_onehot_input,
    columns=onehot_cols,
    dtype=int
)

print("One-Hot Encoded Dataset")

display(df_onehot.head())

print("\nOriginal Categorical Columns:", len(onehot_cols))
print("New Encoded Columns:", df_onehot.shape[1])

## 16. Week 6 Output Summary

In [ ]:
label_col = "role" if "role" in categorical_cols else categorical_cols[0]

df_label = df[label_col].fillna("Unknown").astype(str)

encoder = LabelEncoder()

encoded_values = encoder.fit_transform(df_label)

df_label_encoded = pd.DataFrame({
    label_col: df_label,
    label_col + "_Encoded": encoded_values
})

print("Label Encoding Results")

display(df_label_encoded.head())

print("\nLabel Mapping:")

mapping = dict(
    zip(
        encoder.classes_,
        encoder.transform(encoder.classes_)
    )
)

print(mapping)

## 17. Save and Download the Three Output Files

In [ ]:
print("FINAL FEATURE ENGINEERING SUMMARY")
print("----------------------------------")

print("Original Dataset Shape:", df.shape)

print("Skewness Corrected Dataset Shape:", df_skew.shape)

print("Min-Max Scaled Dataset Shape:", df_minmax.shape)

print("Standard Scaled Dataset Shape:", df_standard.shape)

print("One-Hot Encoded Dataset Shape:", df_onehot.shape)

print("Label Encoded Dataset Shape:", df_label_encoded.shape)

print("\nWeek 6 Feature Engineering Completed Successfully!")

In [ ]:
# Save the three Week 6 outputs as separate CSV files.
# IPL_Skewness_Corrected.csv: full dataset with selected numeric features transformed.
# IPL_Scaled.csv: numeric features after Min-Max scaling.
# IPL_Encoded.csv: selected categorical features after One-Hot Encoding.

df_skew.to_csv("IPL_Skewness_Corrected.csv", index=False)
df_minmax.to_csv("IPL_Scaled.csv", index=False)
df_onehot.to_csv("IPL_Encoded.csv", index=False)

print("Saved files:")
print("1. IPL_Skewness_Corrected.csv")
print("2. IPL_Scaled.csv")
print("3. IPL_Encoded.csv")

from google.colab import files

files.download("IPL_Skewness_Corrected.csv")
files.download("IPL_Scaled.csv")
files.download("IPL_Encoded.csv")

## Interpretation and Notes

- **Skewness:** Values with absolute skewness greater than 1 are selected for Yeo–Johnson transformation. Compare the before-and-after values; do not claim improvement unless the absolute skewness actually decreases.
- **Scaling:** Min-Max scaling places each numerical feature in the 0–1 range. StandardScaler produces features with mean approximately 0 and population standard deviation approximately 1.
- **Encoding:** One-Hot Encoding creates binary indicator columns for the selected categorical variables. Label Encoding assigns integer labels to the selected `role` category; these integers are identifiers, not meaningful rankings.
- **Missing values:** Median imputation is applied to numerical columns for the scaling workflow only. The original Week 03 cleaned dataset remains unchanged.
- **Modeling note:** If these transformations are later used for predictive modeling, fit imputers and scalers on training data only, then apply the fitted transformations to validation/test data to avoid data leakage.

### Expected Week 6 deliverables
- `IPL_Skewness_Corrected.csv`
- `IPL_Scaled.csv`
- `IPL_Encoded.csv`
- Notebook containing executed outputs, statistical tables and plots
- Week 6 report with actual observations from the executed notebook